# 3. Advanced Add-on Exercises: One Possible Solution

This notebook provides **one possible solution** for the exercises in `ex3_advanced_addons.ipynb`.

If you have not yet attempted the exercises yourself, it is recommended to do so before reading this solution.


In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

# Construct src-path and append it to sys.path
src_path = Path.cwd().parent.parent
sys.path.append(str(src_path))

from core.settings import settings  # noqa: F401


## 3.a Solution: Compare Graph Runs & Agent Behaviour

We implement `run_graph_with_summary` by:
- Creating a fresh `State`.
- Running the graph from `PlannerNode`.
- Inspecting the final state to build a compact summary.


In [ ]:
from collections import Counter
from typing import Any, Dict, List

from agents.planner import PlannerNode
from graph import graph
from models.agents import Agents
from models.state import Message, State

async def run_graph_with_summary(user_query: str, enabled_agents: List[Agents]) -> Dict[str, Any]:
    """Run the multi-agent graph once and return a structured summary.

    This implementation follows the suggested steps from the exercise:
    1. Build a fresh State from the user_query and enabled_agents.
    2. Run the graph from PlannerNode.
    3. Inspect plan, messages and replans to create a summary dictionary.
    """
    state = State(
        messages=[Message(content=user_query)],
        user_query=user_query,
        enabled_agents=enabled_agents,
    )

    # Run the graph
    await graph.run(start_node=PlannerNode(), state=state)

    # Plan-related information
    plan = state.plan
    plan_type = plan.type.value if plan is not None else None
    num_plan_steps = len(plan.steps) if plan is not None else 0

    # Replanning statistics (sum of all attempts recorded per step)
    num_replans_total = sum(state.replan_attempts.values()) if state.replan_attempts else 0

    # Messages per agent (creator may be None for initial user message)
    creator_names: List[str] = [m.creator.value if m.creator is not None else "USER" for m in state.messages]
    messages_per_agent = dict(Counter(creator_names))

    final_answer = state.messages[-1].content if state.messages else ""

    return {
        "user_query": user_query,
        "plan_type": plan_type,
        "num_plan_steps": num_plan_steps,
        "num_replans_total": num_replans_total,
        "messages_per_agent": messages_per_agent,
        "final_answer": final_answer,
    }


### Example usage for 3.a (optional)

You can use the following cell to experiment with different queries and inspect the summaries.


In [ ]:
# Example usage (run inside an async context):

# example_queries = [
#     "What are the main drivers of day-ahead electricity prices in Germany today?",
#     "Compare day-ahead electricity price trends between Germany and France over the last week.",
#     "How does increasing solar generation share typically affect intraday price volatility in Germany?",
# ]
#
# enabled = [Agents.WEB_RESEARCHER, Agents.SYNTHESIZER]
#
# summaries = []
# for q in example_queries:
#     summary = await run_graph_with_summary(q, enabled)
#     summaries.append(summary)
#
# for s in summaries:
#     print("=== Query ===")
#     print(s["user_query"])
#     print("Plan type:", s.get("plan_type"))
#     print("# steps:", s.get("num_plan_steps"))
#     print("# replans total:", s.get("num_replans_total"))
#     print("Messages per agent:", s.get("messages_per_agent"))
#     print()


## 3.b Solution: Manual Two-Stage Pipeline (Web Researcher -> Synthesizer)

We now implement `research_and_synthesize` as a small orchestrator that:
- Creates a shared `State`.
- Calls the `web_researcher` agent to populate it with research results.
- Calls the `synthesizer` agent to turn that into a final answer.


In [ ]:
from typing import Tuple

from agents.web_researcher.web_researcher import web_researcher
from agents.synthesizer.synthesizer import synthesizer
from models.agents import Agents
from models.state import Message, State

async def research_and_synthesize(user_query: str) -> Tuple[str, State]:
    """Run a two-stage pipeline: Web Researcher -> Synthesizer.

    1. Create a fresh State with the user_query.
    2. Call web_researcher and append its output to state.messages.
    3. Construct a synthesizer prompt that refers to the accumulated messages.
    4. Call synthesizer and append its output.
    5. Return the synthesizer's answer together with the final state.
    """
    state = State(
        messages=[Message(content=user_query)],
        user_query=user_query,
        enabled_agents=[Agents.WEB_RESEARCHER, Agents.SYNTHESIZER],
    )

    # 1) Web research step
    web_response = await web_researcher.run(user_prompt=user_query, deps=state)
    if isinstance(web_response.output, Message):
        state.messages.append(Message(creator=Agents.WEB_RESEARCHER, content=web_response.output.content))
    else:
        # Fallback: store stringified output
        state.messages.append(Message(creator=Agents.WEB_RESEARCHER, content=str(web_response.output)))

    # 2) Synthesizer step
    synth_prompt = (
        "Using all available messages in the shared state (including web research results), "
        "write a concise, well-structured answer for the user. "
        "Focus on electricity markets and clearly separate key drivers, data points, and uncertainties."
    )

    synth_response = await synthesizer.run(user_prompt=synth_prompt, deps=state)
    if isinstance(synth_response.output, Message):
        final_msg = Message(creator=Agents.SYNTHESIZER, content=synth_response.output.content)
    else:
        final_msg = Message(creator=Agents.SYNTHESIZER, content=str(synth_response.output))

    state.messages.append(final_msg)
    return final_msg.content, state


### Example usage for 3.b (optional)

You can use the following cell to test the two-stage pipeline with different queries.


In [ ]:
# Example usage (run inside an async context):

# query = "Explain today's main drivers of electricity prices in Germany."
# final_answer, final_state = await research_and_synthesize(query)
#
# print("=== Final answer ===")
# print(final_answer)
#
# print("=== Messages in state (creator, first 120 chars) ===")
# for m in final_state.messages:
#     print(m.creator, '-', m.content[:120].replace('\n', ' '), '...')
